In [1]:
!pip install -q sentence-transformers faiss-cpu transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 53.5 MB/s eta 0:00:00


In [2]:
import os
import faiss
import numpy as np

In [3]:
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("src", exist_ok=True)

In [4]:
!ls

data  models  sample_data  src


In [5]:
trading_text = """
RSI stands for Relative Strength Index. It is a momentum indicator used in technical analysis.

MACD stands for Moving Average Convergence Divergence. It helps identify trend changes.

A stop-loss order is used to limit losses by automatically closing a position at a specified level.

A call option gives the buyer the right to purchase an asset at a specified price.

A put option gives the buyer the right to sell an asset at a specified price.

Support is a price level where demand prevents further decline.

Resistance is a price level where selling pressure prevents further price increases.

Swing trading involves holding positions for several days or weeks.

Scalping is a trading strategy that profits from small price movements.

Risk-reward ratio compares the potential profit of a trade to its potential loss.
"""

In [6]:
with open("data/trading_knowledge.txt", "w") as f:
    f.write(trading_text)

print("Knowledge base saved!")

Knowledge base saved!


In [7]:
!ls data

trading_knowledge.txt


In [8]:
with open("data/trading_knowledge.txt", "r") as f:
    text = f.read()

chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]

print("Total Chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:")
    print(chunk)

Total Chunks: 10

Chunk 1:
RSI stands for Relative Strength Index. It is a momentum indicator used in technical analysis.

Chunk 2:
MACD stands for Moving Average Convergence Divergence. It helps identify trend changes.

Chunk 3:
A stop-loss order is used to limit losses by automatically closing a position at a specified level.

Chunk 4:
A call option gives the buyer the right to purchase an asset at a specified price.

Chunk 5:
A put option gives the buyer the right to sell an asset at a specified price.

Chunk 6:
Support is a price level where demand prevents further decline.

Chunk 7:
Resistance is a price level where selling pressure prevents further price increases.

Chunk 8:
Swing trading involves holding positions for several days or weeks.

Chunk 9:
Scalping is a trading strategy that profits from small price movements.

Chunk 10:
Risk-reward ratio compares the potential profit of a trade to its potential loss.


In [9]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(chunks)

print("Embedding Shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Shape: (10, 384)


In [10]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

print("Documents Indexed:", index.ntotal)

Documents Indexed: 10


In [11]:
faiss.write_index(
    index,
    "models/trading_index.faiss"
)

print("FAISS index saved!")

FAISS index saved!


In [12]:
!ls models

trading_index.faiss


In [13]:
query = "What is RSI?"

query_embedding = embedding_model.encode([query])

D, I = index.search(
    np.array(query_embedding).astype("float32"),
    k=3
)

print("Top Matches:\n")

for idx in I[0]:
    print(chunks[idx])
    print("-" * 50)

Top Matches:

RSI stands for Relative Strength Index. It is a momentum indicator used in technical analysis.
--------------------------------------------------
MACD stands for Moving Average Convergence Divergence. It helps identify trend changes.
--------------------------------------------------
A stop-loss order is used to limit losses by automatically closing a position at a specified level.
--------------------------------------------------


In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer_gen = AutoTokenizer.from_pretrained(model_name)
model_gen = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 Loaded Successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 Loaded Successfully!


In [15]:
def retrieve_context(query, k=3):
    query_embedding = embedding_model.encode([query])

    D, I = index.search(
        np.array(query_embedding).astype("float32"),
        k
    )

    contexts = [chunks[idx] for idx in I[0]]

    return "\n".join(contexts)

In [16]:
def ask_trading_bot(question):

    context = retrieve_context(question)

    prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer_gen(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model_gen.generate(
        **inputs,
        max_new_tokens=50
    )

    answer = tokenizer_gen.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [17]:
print(ask_trading_bot("What is RSI?"))

Relative Strength Index


In [18]:
print(ask_trading_bot("What is MACD?"))

Moving Average Convergence Divergence


In [19]:
print(ask_trading_bot("What is swing trading?"))

holding positions for several days or weeks


In [20]:
faiss.write_index(
    index,
    "models/trading_index.faiss"
)

In [21]:
!ls models

trading_index.faiss


In [22]:
requirements = """
sentence-transformers
faiss-cpu
transformers
torch
streamlit
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created")

requirements.txt created


In [23]:
app_code = '''
import streamlit as st

st.title("📈 Trading RAG Assistant")

st.write("Trading AI Assistant Project")

question = st.text_input("Ask a trading question")

if question:
    st.write("Answer:")
    st.write("Connect RAG pipeline here")
'''

with open("src/app.py", "w") as f:
    f.write(app_code)

print("app.py created")

app.py created


In [24]:
!find . -maxdepth 2

.
./.config
./.config/.last_update_check.json
./.config/.last_survey_prompt.yaml
./.config/gce
./.config/.last_opt_in_prompt.yaml
./.config/config_sentinel
./.config/default_configs.db
./.config/logs
./.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
./.config/active_config
./.config/configurations
./src
./src/app.py
./models
./models/trading_index.faiss
./requirements.txt
./data
./data/trading_knowledge.txt
./sample_data
./sample_data/anscombe.json
./sample_data/README.md
./sample_data/mnist_train_small.csv
./sample_data/mnist_test.csv
./sample_data/california_housing_test.csv
./sample_data/california_housing_train.csv


In [25]:
from google.colab import files
files.download("data/trading_knowledge.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
from google.colab import files
files.download("models/trading_index.faiss")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
from google.colab import files
files.download("src/app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
from google.colab import files
files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>